# Question 1

To be able to use this model, we need to know the name of the input and output nodes.

What's the name of the output:

output

In [12]:
import onnxruntime as ort

In [14]:
onnx_model_path = "homework_9/hair_classifier_v1.onnx"
session = ort.InferenceSession(onnx_model_path, providers=["CPUExecutionProvider"])
# 取出模型的输入、输出信息列表
inputs = session.get_inputs() 
outputs = session.get_outputs() 

for i in inputs:
    print(i.name, i.shape, i.type)


for o in outputs:
    print(o.name, o.shape, o.type)


# 取第一个输入的名字、第一个输出的名字。后面推理时要按这些名字传数据、取结果。
input_name = inputs[0].name
output_name = outputs[0].name

input ['s77', 3, 200, 200] tensor(float)
output ['s77', 1] tensor(float)


In [3]:
# 从网络下载一张图片，并把它调整到模型需要的输入尺寸。它是图像预处理流水线的一部分，通常用在推理之前
import numpy as np
from io import BytesIO
from urllib import request

from PIL import Image

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

# Question 2: Target size

In [ ]:
Let's download and resize this image:

https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

Based on the previous homework, what should be the target size for the image?

200x200

# Question 3

In [19]:
from tensorflow.keras.applications.xception import preprocess_input
url = 'https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg'
img = download_image(url)
img = prepare_image(img, (200, 200)) 
x = np.array(img, dtype=np.float32)
x = preprocess_input(x)
print(x[0, 0, 0])  # 第一个像素的 R 通道

-0.52156866


# Question 4

- Now let's apply this model to this image. What's the output of the model?

In [22]:
from keras_image_helper import create_preprocessor
# 创建一个基于 Xception 模型的图像预处理流程，并把输入图像统一调整到 200×200 大小
preprocessor = create_preprocessor('xception', target_size=(200, 200))

url = 'https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg'
X = preprocessor.from_url(url)
print(X.shape)

(1, 200, 200, 3)


In [23]:
# NHWC -> NCHW
X = np.transpose(X, (0, 3, 1, 2))
print(X.shape)

(1, 3, 200, 200)


In [28]:
result = session.run([output_name], {input_name: X})
print(f'result={result}')
# 取出 logit（标量）
logit = float(result[0][0][0])
print(f'logit = {logit}')

# sigmoid
prob = 1 / (1 + np.exp(-logit))
print(f'probability = {prob}')

# 按 0.5 阈值判断类别
predicted = 1 if prob > 0.5 else 0
print(f'predicted class = {predicted}')



result=[array([[-0.20132379]], dtype=float32)]
logit = -0.20132379233837128
probability = 0.4498383638070855
predicted class = 0


#  Question 5

In [ ]:
208 Mb

# Question 6

In [ ]:
0.10